# WIP GenAI Evaluation Framework in MLflow 3.14

| 🚨 Consider a WIP until this message goes away... 🚨 |
|-|

[MLflow 3.14.0's source code](https://github.com/mlflow/mlflow/tree/v3.14.0)

# Set Up Learning Environment

In [0]:
%pip install -qU "mlflow-skinny[databricks]>=3.14.0" "databricks-agents>=1.11.0" databricks-langchain
%restart_python

In [0]:
import mlflow

displayHTML(f"MLflow version: <b>{mlflow.__version__}</b>")

In [0]:
%pip show databricks-agents

# mlflow.genai.evaluate


[mlflow.genai.evaluate](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html#mlflow.genai.evaluate)'s signature ([GitHub](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/base.py#L55-L60)):

<br>

```
evaluate(
    data: 'EvaluationDatasetTypes',
    scorers: list[mlflow.genai.scorers.base.Scorer],
    predict_fn: Optional[Callable[..., Any]] = None,
    model_id: str | None = None
) -> 'EvaluationResult'
```

Evaluates the performance of a generative AI model/application using the specified data and scorers:

* `data: 'EvaluationDatasetTypes'` (we'll talk about it later)
* `scorers: list[mlflow.genai.scorers.base.Scorer]` with built-in scorers provided by MLflow and custom scorers (we'll talk about it later, too)
* `predict_fn: Optional[Callable[..., Any]] = None` is for the Gen AI app to be evaluated
* `model_id: str | None = None` is for an AI model

Returns `EvaluationResult` with metrics and detailed per-row assessments (discussed later in this notebook)

Evaluates a model's performance on a given dataset using various scoring criteria.


## Advanced Features on Databricks

Certain advanced features of this function are only supported on Databricks.

The tracking URI must be set to Databricks to use these features.

⚠️ **NOTE**: I've got no idea what these advanced features are.


## Measurable Variables

The "variables" of agentic applications to be measured using [MLflow's evaluation and monitoring capabilities](https://mlflow.org/docs/latest/genai/eval-monitor/):

* **Language Models** (LM) chosen (e.g., [ChatGPT](https://openai.com/index/chatgpt/), [Claude](https://platform.claude.com/docs/en/about-claude/models/overview))
* **Model Versions** (e.g., [OpenAI's GPT-5.5](https://openai.com/index/introducing-gpt-5-5/), [Claude Opus 4.8](https://www.anthropic.com/news/claude-opus-4-8))
* **LLM (Hyper)Parameters** (e.g., temperature, top-k, top-p (nucleus sampling), max tokens)
* **Prompt Versions**



## FIXME Jupyter Lab

🚨 **NOTE**: I really wanted to show you how to run this notebook outside Databricks using [JupyterLab: A Next-Generation Notebook Interface](https://jupyter.org/), but ran into some issues with the very first python cell and gave up...for now 🤞

<br>

```shell
uvx --with jupyterlab jupyter lab
```

## mlflow.genai

[mlflow.genai](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html)

In [0]:
from mlflow.genai import evaluate

help(mlflow.genai.evaluate)

# EvaluationDatasetTypes

[EvaluationDatasetTypes](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/utils.py#L30-L52) type alias lists the acceptable types of the `data` that `mlflow.genai.evaluate` runs `scorers` on.

<br>

```
evaluate(
    data: 'EvaluationDatasetTypes',
    ...
) -> 'EvaluationResult'
```

<br>

```py
EvaluationDatasetTypes = (
    pd.DataFrame
    | pyspark.sql.dataframe.DataFrame
    | list[dict]
    | list[Trace]
    | mlflow.genai.datasets.EvaluationDataset
    | mlflow.entities.evaluation_dataset.EvaluationDataset
    | mlflow.genai.simulators.ConversationSimulator
    | None
)
```

`EvaluationDatasetTypes` cannot be imported directly since it is guarded by [TYPE_CHECKING](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/utils.py#L25-L52) flag.

An interesting part is the [try-except](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/utils.py#L30-L52) block.
Unless MLflow runs in a PySpark runtime (e.g., Databricks), `pyspark.sql.dataframe.DataFrame` won't be included among the allowed datasets.

In [0]:
from typing import TYPE_CHECKING

print(TYPE_CHECKING)

## EvaluationDataset

Provides a unified interface for evaluation datasets.

Evaluation Dataset | Description
-|-
`mlflow.entities.evaluation_dataset.EvaluationDataset` | Standard MLflow evaluation datasets (backed by MLflow's tracking store)
`databricks.agents.datasets.Dataset` | Databricks-managed datasets (backed by Unity Catalog tables)

The `databricks-agents` package is required to use `mlflow.genai.datasets`.

In [0]:
from mlflow.genai.datasets import EvaluationDataset

help(EvaluationDataset)

In [0]:
import mlflow

experiments = mlflow.search_experiments()
print(f"Number of experiments: {len(experiments)}")

In [0]:
import pandas as pd

experiments_list = [exp.__dict__ for exp in experiments]
display(pd.DataFrame(experiments_list))

In [0]:
from mlflow.tracking.fluent import _get_experiment_id

current_notebook_experiment_id = _get_experiment_id()

In [0]:
from mlflow.genai.datasets import search_datasets

# Returns ALL datasets in your tracking server.
# In Databricks, at least one `experiment_id` filter must be provided.
# Listing across all experiments is not supported.

experiment_ids = [current_notebook_experiment_id]
search_datasets(experiment_ids=experiment_ids)

In [0]:
%sql

CREATE CATALOG IF NOT EXISTS jacek_laskowski;
CREATE SCHEMA IF NOT EXISTS jacek_laskowski.mlflow;

In [0]:
%sql

DROP TABLE IF EXISTS jacek_laskowski.mlflow.qa_dataset

In [0]:
from mlflow.genai.datasets import create_dataset

dataset = create_dataset(
    # In Databricks, the dataset name must be a UC table name in the format: catalog.schema.table
    name="jacek_laskowski.mlflow.qa_dataset",
    experiment_id=current_notebook_experiment_id,
    # Tags are not supported in Databricks environments.
    # Tags are managed through Unity Catalog.
    # tags={
    #     "version": "1.0",
    #     "purpose": "regression_testing",
    #     "model": "gpt-4",
    #     "team": "ml-platform",
    # },
)
print(f"Created dataset: {dataset.dataset_id}")


# Scorer

[mlflow.genai.scorers.base.Scorer](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/base.py#L218-L1120) is a [pydantic.BaseModel](https://pydantic.dev/docs/validation/latest/concepts/models/) with the following properties:

* `name` (required)
* `aggregations`
* `description`
* `kind` ([ScorerKind](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/base.py#L47-L54) enum)

Aggregations can be the literals or a custom function (of `int`s and `float`s):

* `_AggregationFunc: TypeAlias = Callable[[list[int | float]], float]`
* ```_AggregationType: TypeAlias = (Literal["min", "max", "mean", "median", "variance", "p90"] | _AggregationFunc)```

There are [built-in scorers](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/__init__.py) (provided by MLflow) and custom scorers.

🚨 [Report this "discrepancy"](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/__init__.py#L118).

<br>

```py
from mlflow.genai.scorers import Correctness, Safety
```

All the built-in scorers inherit from [BuiltInScorer(Judge)](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/builtin_scorers.py#L313C1-L388) class.

[Judge](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/judges/base.py#L53C1-L137) class is a...[Scorer](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/genai/scorers/base.py#L218-L1120).

You can invoke scorers directly with a single input (for testing), or pass them to `mlflow.genai.evaluate` for running full evaluation on a dataset.

## Scorers are Functions (Callables)

<br>

```py
def __call__(
    self,
    *,
    inputs: Any = None,
    outputs: Any = None,
    expectations: dict[str, Any] | None = None,
    trace: Trace | None = None,
    session: list[Trace] | None = None,
) -> int | float | bool | str | Feedback | list[Feedback]:
    raise NotImplementedError("Implementation of __call__ is required for Scorer class")
```

Scorers raise a `NotImplementedError` by default.

User-defined scorers don't need to have all the parameters defined from the base signature.
You can define a custom scorer with only the parameters you need.

When executed, scorers return one of the following:

* `int`
* `float`
* `bool`
* `str`
* `mlflow.entities.assessment.Feedback` (discussed later)
* `list[mlflow.entities.assessment.Feedback]`

In [0]:
from mlflow.genai import Scorer

help(Scorer.__call__)


### Automatic Trace Evaluation

Scorers can be registered with a MLflow server (`ScorerStoreRegistry`) for automatic trace evaluation in the specified experiment (using `Scorer.register` method).

<br>

```py
def register(
    self,
    *,
    name: str | None = None,
    experiment_id: str | None = None) -> "Scorer":
    ...
```

`experiment_id` is the ID of the MLflow experiment to register the scorer for.
If undefined (`None`), uses the currently active experiment.

Once registered, the scorer can be `start`ed to begin evaluating traces automatically.

The scorer registration can be `update`d and eventually `stop`ped (deactivated).

In [0]:
from mlflow.genai.scorers import Correctness

help(Correctness.register)

In [0]:
import mlflow
from mlflow.genai.scorers import RelevanceToQuery

# The same name must not be used twice
# hence try-except block for reproducibility

scorer_name = "relevance_scorer"
try:
    registered_scorer = RelevanceToQuery().register(name=scorer_name)
    print(registered_scorer)
except ValueError as ve:
    print(f"Exception: {ve}")

In [0]:
from mlflow.genai.scorers import scorer

@scorer
def custom_length_check(outputs) -> bool:
    return len(outputs) > 100

# The same name must not be used twice
# hence try-except block for reproducibility

custom_scorer_name = "output_length_checker"
try:
    registered_custom_scorer = custom_length_check.register(
        name=custom_scorer_name,
        experiment_id=current_notebook_experiment_id,
    )
    print(registered_custom_scorer)
except ValueError as ve:
    print(f"Exception: {ve}")

In [0]:
custom_length_check


### Mosaic AI Agent Framework SDK

In Databricks workspace, the [databricks-agents](https://pypi.org/project/databricks-agents/) package is required to register scorers.

In [0]:
%pip show databricks-agents

In [0]:
import mlflow

# Get the current tracking uri
tracking_uri = mlflow.get_tracking_uri()
print(f"Current tracking uri: {tracking_uri}")

assert tracking_uri == "databricks"

# Get the scorer store for the tracking uri
scorer_store = mlflow.genai.scorers.registry._get_scorer_store(tracking_uri=tracking_uri)
print(f"Scorer store: {scorer_store}")

registered_scorer = scorer_store.get_scorer(name=scorer_name, experiment_id=current_notebook_experiment_id)
print(f"Registered scorer: {registered_scorer}")

In [0]:
# All registered scorers in Databricks will have databricks backend assigned

from mlflow.genai.scorers.base import SCORER_BACKEND_DATABRICKS

print(SCORER_BACKEND_DATABRICKS)


## Built-In Scorers

**Built-in Scorers** inherit from [BuiltInScorer](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L314-L389):

* [Correctness](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1699-L1891) evaluates whether the model's response supports the expected facts or response.
* [Safety](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1593-L1695) ensures that the agent's responses do not contain harmful, offensive, or toxic content.
* [a few others](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/__init__.py#L30-L53)

Properties:
* `name`
* `required_columns`
* `inference_params` (optional)
* `instructions` of what this scorer evaluates

Operators:
* `model_dump` for serialization (using `pydantic.BaseModel.model_dump` in JSON mode)
* `model_validate` for deserialization

In [0]:
import pydantic

print(pydantic.__version__)

In [0]:
import pydantic

help(pydantic.BaseModel.model_dump)

## Examples

`expectations` field can only be a dict with `expected_response` or `expected_facts`.

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

scorer = Correctness(name="my_correctness")

print(f"description:\n{scorer.description}")
print("")
print(f"instructions:\n{scorer.instructions}")

In [0]:
# Scorers are callables
assessment = scorer(
    # required
    # https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1771
    inputs={
        "question": "What is the difference between reduceByKey and groupByKey in Spark?"
    },
    # required
    # https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1771
    outputs=(
        "reduceByKey aggregates data before shuffling, whereas groupByKey "
        "shuffles all data, making reduceByKey more efficient."
    ),
    # expectations: dict[str, Any] | None = None,
    expectations={
        "expected_response": (
            "reduceByKey aggregates data before shuffling"
            "groupByKey shuffles all data"
        )
    },
)
print(f"assessment:\n{assessment}")

In [0]:
import mlflow
from mlflow.genai.scorers import Correctness

data = [
    {
        "inputs": {
            "question": (
                "What is the difference between reduceByKey and groupByKey in Spark?"
            )
        },
        "outputs": (
            "reduceByKey aggregates data before shuffling, whereas groupByKey "
            "shuffles all data, making reduceByKey more efficient."
        ),
        "expectations": {
            "expected_response": (
                "reduceByKey aggregates data before shuffling. groupByKey shuffles all data"
            ),
        },
    }
]
result = mlflow.genai.evaluate(data=data, scorers=[Correctness()])
print(result)

## Correctness scorer

Evaluates correctness of the response against expectations (_ground truth_)

`Correctness` is a built-in scorer ([BuiltInScorer](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L314-L389)).

Properties:
* `name` always **correctness**
* `model` (discussed later)
* `required_columns` are "inputs", "outputs"

Correctness is a LLM judge that uses a judge model for evaluation (discussed later).

In [0]:
# Part of BuiltInScorer.model_validate

from mlflow.genai.scorers import builtin_scorers

correctness_class = getattr(builtin_scorers, 'Correctness')
print(correctness_class)

In [0]:
from mlflow.genai.scorers import Correctness

# Note the Args section for the model argument (discussed later)
help(Correctness)


### LLM Judges

`Correctness` is one of the built-in scorers that accepts a LLM judge model for execution.

<br>

```
 |      model: Judge model to use. Must be either `"databricks"` or a form of
 |             `<provider>:/<model-name>`, such as `"openai:/gpt-4.1-mini"`,
 |             `"anthropic:/claude-3.5-sonnet-20240620"`. MLflow natively supports
 |             `["openai", "anthropic", "bedrock", "mistral"]`, and more providers are supported
 |             through `LiteLLM <https://docs.litellm.ai/docs/providers>`_.
 |             Default model depends on the ``MLFLOW_GENAI_JUDGE_DEFAULT_MODEL`` environment
 |             variable and the tracking URI setup:
 |
 |             * If ``MLFLOW_GENAI_JUDGE_DEFAULT_MODEL`` is set, that value is used.
 |             * Databricks tracking URI: `databricks`
 |             * Otherwise: `openai:/gpt-4.1-mini`.
```

Other LLM judge model-based built-in scorers (that use `mlflow.genai.judges.utils.invoke_judge_model` for execution):

* `Equivalence` for semantic equivalence.
* `InstructionsJudge` for trace evaluation based on user-provided instructions.
* `RetrievalRelevance` that measures whether each chunk is relevant to the input request. Uses natural language instructions to guide evaluation, making it flexible for various assessment criteria.
* Others that use the LLM judges (`mlflow/genai/judges/builtin.py` methods) and return `Feedback`:
    * `meets_guidelines`
    * `is_tool_call_efficient` (experimental since 3.8.0)
    * `is_tool_call_correct` (experimental since 3.8.0)
    * `is_safe`
    * `is_grounded`
    * `is_correct`
    * `is_context_sufficient`
    * `is_context_relevant`

In [0]:
# the default LLM judge model is used when no model is passed explicitly.
# model_uri is the full model URI (e.g., "openai:/gpt-4.1-mini", "databricks").
# - The default Databricks judge ("databricks")
# - AI Gateway (Databricks model serving) for model provider endpoints ("<provider>:/<model-name>")
# - LiteLLM-supported models (provided LiteLLM is installed)

from mlflow.genai.judges.utils import get_default_model

print(get_default_model())

In [0]:
%pip show litellm

In [0]:
from mlflow.genai.scorers import Correctness

help(Correctness.__call__)


`Correctness` returns a [mlflow.entities.assessment.Feedback](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment.py#L193-L363) (which is a [Assessment](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment.py#L34-L186) FWIW).

In [0]:
help(mlflow.entities.assessment.Feedback)


Among the properties of `Feedback` is `AssessmentSource` with [AssessmentSourceType](https://github.com/mlflow/mlflow/blob/v3.13.0/mlflow/entities/assessment_source.py#L102C7-L194):

- `HUMAN`: Assessment performed by a human evaluator
- `LLM_JUDGE`: Assessment performed by an LLM-as-a-judge (e.g., GPT-4)
- `CODE`: Assessment performed by deterministic code/heuristics
- `SOURCE_TYPE_UNSPECIFIED`: Default when source type is not specified

In [0]:
yes_or_no = Correctness()

# builtin_scorer_class is required for built-in scorer deserialization using model_validate class method
assert yes_or_no.model_dump()['builtin_scorer_class'] == Correctness.__name__

display(yes_or_no.model_dump())

In [0]:
print(yes_or_no.instructions)

In [0]:
# Type of the feedback value

yes_or_no.feedback_value_type

In [0]:
# the input fields for this judge

input_fields = [
    {**field.__dict__, "value_type": str(field.value_type)}
    for field in yes_or_no.get_input_fields()
]
display(input_fields)

In [0]:
# Correctness.validate_columns validates the input columns (the input fields above)

# Checks required_columns first
# {"inputs", "outputs"}

print(f"required_columns: {yes_or_no.required_columns}")

columns: set[str] = set()
try:
    yes_or_no.validate_columns(columns)
except Exception as e:
    print(e)

# Then, Correctness-specific ones
# {"expectations/expected_response", "expectations/expected_facts"}

columns = {'outputs', 'inputs'}
try:
    yes_or_no.validate_columns(columns)
except Exception as e:
    print(e)


<br>

```py
def __call__(
    self,
    *,
    inputs: dict[str, Any] | None = None,
    outputs: Any | None = None,
    expectations: dict[str, Any] | None = None,
    trace: Trace | None = None,
) -> Feedback:
    ...
```

[Correctness.\_\_call__](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/scorers/builtin_scorers.py#L1827-L1891)

# EvaluationResult

[EvaluationResult](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/evaluation/entities.py#L255-L332) is a Python `@dataclass`.

In [0]:
display(result.result_df)

# How to Use

There are ~~three~~ four different ways to use this function (see [mlflow.genai.evaluate](https://mlflow.org/docs/latest/api_reference/python_api/mlflow.genai.html#mlflow.genai.evaluate)):

1. Use Traces to evaluate the model/application.
1. Use DataFrame or dictionary with “inputs”, “outputs”, “expectations” columns.
1. Pass `predict_fn` and input samples (and optionally expectations).
1. Pass a ConversationSimulator for multi-turn evaluation.

## Traces

**MLflow Trace Objects** contain the execution to evaluate.

When provided, scorers will use them to extract necessary data to evaluate

## ConversationSimulator

<br>

```py
from mlflow.genai.simulators import ConversationSimulator
```

Available as `mlflow.genai.ConversationSimulator` (through [mlflow/genai/\_\_init__.py](https://github.com/mlflow/mlflow/blob/v3.14.0/mlflow/genai/__init__.py#L95))

### Purpose by Claude

Generates multi-turn synthetic conversations between an LLM-simulated "user" and your agent's `predict_fn`, tracing every turn in MLflow.

Designed to plug directly into `mlflow.genai.evaluate(data=simulator, ...)` as a data source, so you can run scorers (e.g. `Safety`, `ConversationalSafety`) over realistic multi-turn dialogues instead of hand-writing them.

---

prompt: "brief me about ConversationSimulator in mlflow.genai.simulators"

In [0]:
from mlflow.genai import ConversationSimulator

help(ConversationSimulator)

## Example: Automatic Trace Evaluation

There are a few scorers registered earlier in this notebook:

1. `RelevanceToQuery`
1. `output_length_checker`

Go to **Judges** in the MLflow UI for this notebook experiment.

You're now going to autolog an LLM application.
This will be a [simple LangChain agent with a custom tool](https://docs.langchain.com/oss/python/langchain/overview#create-an-agent).

In [0]:
%pip show deepagents

In [0]:
import mlflow

mlflow.langchain.autolog()

In [0]:
%pip show databricks-langchain

In [0]:
from databricks_langchain import ChatDatabricks

LLM_ENDPOINT_NAME = "databricks-claude-opus-4-8"

databricks_chat_model = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

In [0]:
help(registered_scorer.start)

In [0]:
# Start scorer with 50% sampling rate

from mlflow.genai.scorers import ScorerSamplingConfig

active_scorer = registered_scorer.start(sampling_config=ScorerSamplingConfig(sample_rate=1))
print(f"Scorer is evaluating {active_scorer.sample_rate * 100}% of traces")

In [0]:
# Create a simple LangChain agent with a custom tool
# https://docs.langchain.com/oss/python/langchain/overview#create-an-agent

from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=databricks_chat_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What's the weather in San Francisco?"}]}
)
print(result["messages"][-1].content_blocks)